# AdamW over a pytree with a weight-decay mask

Write one AdamW update for an arbitrary parameter pytree, in the functional style JAX uses: the optimizer state goes in and comes back out.

Write `adamw_update(params, grads, state, lr, b1, b2, eps, wd)`:

- `params`, `grads`: pytrees (nested dicts) of float arrays.
- `state`: dict `{'m': pytree, 'v': pytree, 'count': int}`. `m` and `v` match `params`; `count` is the number of steps taken so far (0 before the first step).

With $t$ = `count + 1`, for each leaf:

$$m \leftarrow \beta_1 m + (1-\beta_1) g, \qquad v \leftarrow \beta_2 v + (1-\beta_2) g^2$$

$$\hat m = \frac{m}{1-\beta_1^t}, \qquad \hat v = \frac{v}{1-\beta_2^t}, \qquad p \leftarrow p - \eta\left(\frac{\hat m}{\sqrt{\hat v} + \epsilon} + \lambda_p\, p\right)$$

Weight decay is **decoupled** (it is not added to $g$, so it skips the moment estimates) and **masked**: $\lambda_p = $ `wd` for leaves with `ndim >= 2` (weight matrices, embeddings) and $\lambda_p = 0$ for leaves with `ndim < 2` (biases, norm scales).

Return `(new_params, new_state)` with `new_state = {'m': ..., 'v': ..., 'count': count + 1}`. float32 leaves; `count` may be a Python int or a 0-d int array.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def adamw_update(params, grads, state, lr, b1, b2, eps, wd):
    """state = {'m', 'v', 'count'}. Return (new_params, new_state)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__dict__": [["w", {"__nd__": [[1.0]], "dtype": "float64"}], ["b", {"__nd__": [1.0], "dtype": "float64"}]]}, {"__dict__": [["w", {"__nd__": [[0.5]], "dtype": "float64"}], ["b", {"__nd__": [0.5], "dtype": "float64"}]]}, {"__dict__": [["m", {"__dict__": [["w", {"__nd__": [[0.0]], "dtype": "float64"}], ["b", {"__nd__": [0.0], "dtype": "float64"}]]}], ["v", {"__dict__": [["w", {"__nd__": [[0.0]], "dtype": "float64"}], ["b", {"__nd__": [0.0], "dtype": "float64"}]]}], ["count", 0]]}, 0.1, 0.9, 0.999, 1e-08, 0.1], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": [0.9000006914138794], "dtype": "float32"}], ["w", {"__nd__": [[0.8900006413459778]], "dtype": "float32"}]]}, {"__dict__": [["m", {"__dict__": [["b", {"__nd__": [0.05000000074505806], "dtype": "float32"}], ["w", {"__nd__": [[0.05000000074505806]], "dtype": "float32"}]]}], ["v", {"__dict__": [["b", {"__nd__": [0.0002500000118743628], "dtype": "float32"}], ["w", {"__nd__": [[0.0002500000118743628]], "dtype": "float32"}]]}], ["count", {"__nd__": 1, "dtype": "int32"}]]}]}}, {"a": [{"__dict__": [["w", {"__nd__": [[2.0, -2.0]], "dtype": "float64"}]]}, {"__dict__": [["w", {"__nd__": [[0.0, 0.0]], "dtype": "float64"}]]}, {"__dict__": [["m", {"__dict__": [["w", {"__nd__": [[0.0, 0.0]], "dtype": "float64"}]]}], ["v", {"__dict__": [["w", {"__nd__": [[0.0, 0.0]], "dtype": "float64"}]]}], ["count", 0]]}, 0.5, 0.9, 0.999, 1e-08, 0.1], "e": {"__tup__": [{"__dict__": [["w", {"__nd__": [[1.899999976158142, -1.899999976158142]], "dtype": "float32"}]]}, {"__dict__": [["m", {"__dict__": [["w", {"__nd__": [[0.0, 0.0]], "dtype": "float32"}]]}], ["v", {"__dict__": [["w", {"__nd__": [[0.0, 0.0]], "dtype": "float32"}]]}], ["count", {"__nd__": 1, "dtype": "int32"}]]}]}}, {"a": [{"__dict__": [["w", {"__nd__": [[-0.43234000554110225, -1.130096771890423], [0.6738436579279797, -1.1078145982748595], [2.013915583091144, 0.9241120367011457]], "dtype": "float64"}], ["b", {"__nd__": [-0.3592629380835823, 0.5705157312751826], "dtype": "float64"}], ["ln", {"__dict__": [["scale", {"__nd__": [1.0, 1.0], "dtype": "float64"}]]}]]}, {"__dict__": [["w", {"__nd__": [[1.6115890898926153, 2.833969406582865], [-0.9232646653032858, 1.0650737311362373], [0.5177406291981792, -0.27955103506860146]], "dtype": "float64"}], ["b", {"__nd__": [1.092198183082093, 0.506774300112912], "dtype": "float64"}], ["ln", {"__dict__": [["scale", {"__nd__": [1.0, 1.0], "dtype": "float64"}]]}]]}, {"__dict__": [["m", {"__dict__": [["w", {"__nd__": [[0.0, 0.0], [0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}], ["b", {"__nd__": [0.0, 0.0], "dtype": "float64"}], ["ln", {"__dict__": [["scale", {"__nd__": [0.0, 0.0], "dtype": "float64"}]]}]]}], ["v", {"__dict__": [["w", {"__nd__": [[0.0, 0.0], [0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}], ["b", {"__nd__": [0.0, 0.0], "dtype": "float64"}], ["ln", {"__dict__": [["scale", {"__nd__": [0.0, 0.0], "dtype": "float64"}]]}]]}], ["count", 0]]}, 0.001, 0.9, 0.999, 1e-08, 0.01], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": [-0.360262930393219, 0.5695157647132874], "dtype": "float32"}], ["ln", {"__dict__": [["scale", {"__nd__": [0.9990000128746033, 0.9990000128746033], "dtype": "float32"}]]}], ["w", {"__nd__": [[-0.43333566188812256, -1.1310855150222778], [0.6748369336128235, -1.1088035106658936], [2.012895345687866, 0.9251027703285217]], "dtype": "float32"}]]}, {"__dict__": [["m", {"__dict__": [["b", {"__nd__": [0.10921981185674667, 0.050677429884672165], "dtype": "float32"}], ["ln", {"__dict__": [["scale", {"__nd__": [0.10000000149011612, 0.10000000149011612], "dtype": "float32"}]]}], ["w", {"__nd__": [[0.16115890443325043, 0.2833969295024872], [-0.09232646971940994, 0.10650737583637238], [0.05177406221628189, -0.027955103665590286]], "dtype": "float32"}]]}], ["v", {"__dict__": [["b", {"__nd__": [0.0011928968597203493, 0.00025682023260742426], "dtype": "float32"}], ["ln", {"__dict__": [["scale", {"__nd__": [0.0010000000474974513, 0.0010000000474974513], "dtype": "float32"}]]}], ["w", {"__nd__": [[0.0025972193107008934, 0.008031382225453854], [0.0008524177246727049, 0.0011343820951879025], [0.00026805532979778945, 7.814878335921094e-05]], "dtype": "float32"}]]}], ["count", {"__nd__": 1, "dtype": "int32"}]]}]}}, {"a": [{"__dict__": [["w", {"__nd__": [[1.0762408357856637, -0.5258990543279097], [-0.0024914766372042224, 0.39496114690847944], [0.019602356389466728, 0.016572438548405812]], "dtype": "float64"}], ["b", {"__nd__": [-0.7684164980499638, 0.1263793886240963], "dtype": "float64"}], ["ln", {"__dict__": [["scale", {"__nd__": [1.0, 1.0], "dtype": "float64"}]]}]]}, {"__dict__": [["w", {"__nd__": [[0.1876463981373617, -0.061309977606480055], [-0.01474794159135377, 0.6964605052716821], [-0.6586030890222689, 0.696522792827198]], "dtype": "float64"}], ["b", {"__nd__": [-0.15348328159050995, 0.05616608851442485], "dtype": "float64"}], ["ln", {"__dict__": [["scale", {"__nd__": [1.0, 1.0], "dtype": "float64"}]]}]]}, {"__dict__": [["m", {"__dict__": [["w", {"__nd__": [[-0.19089152810968965, 0.8991820885212743], [0.7605699179623647, 0.006135799849364996], [-0.4385210147239564, 1.8563997563694796]], "dtype": "float64"}], ["b", {"__nd__": [0.5390181728122847, -1.146409792878106], "dtype": "float64"}], ["ln", {"__dict__": [["scale", {"__nd__": [1.0, 1.0], "dtype": "float64"}]]}]]}], ["v", {"__dict__": [["w", {"__nd__": [[0.004381248974623983, 0.11233602594477322], [0.7218949488881095, 0.22659575841286586], [0.7393272287606538, 1.3381937236315908]], "dtype": "float64"}], ["b", {"__nd__": [1.9925427794894295, 1.4141370735336276], "dtype": "float64"}], ["ln", {"__dict__": [["scale", {"__nd__": [0.5227755769828212, 0.5143370495240375], "dtype": "float64"}]]}]]}], ["count", 7]]}, 0.01, 0.9, 0.95, 1e-06, 0.1], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": [-0.7718935608863831, 0.13539725542068481], "dtype": "float32"}], ["ln", {"__dict__": [["scale", {"__nd__": [0.9862223267555237, 0.9861201643943787], "dtype": "float32"}]]}], ["w", {"__nd__": [[1.0954208374023438, -0.5503943562507629], [-0.010890694335103035, 0.39300164580345154], [0.025095883756875992, 0.0009800270199775696]], "dtype": "float32"}]]}, {"__dict__": [["m", {"__dict__": [["b", {"__nd__": [0.4697680175304413, -1.0261521339416504], "dtype": "float32"}], ["ln", {"__dict__": [["scale", {"__nd__": [1.0, 1.0], "dtype": "float32"}]]}], ["w", {"__nd__": [[-0.15303772687911987, 0.8031328320503235], [0.6830381155014038, 0.07516827434301376], [-0.4605292081832886, 1.7404121160507202]], "dtype": "float32"}]]}], ["v", {"__dict__": [["b", {"__nd__": [1.8940935134887695, 1.3435879945755005], "dtype": "float32"}], ["ln", {"__dict__": [["scale", {"__nd__": [0.5466368198394775, 0.5386201739311218], "dtype": "float32"}]]}], ["w", {"__nd__": [[0.005922745447605848, 0.10690717399120331], [0.6858110427856445, 0.23951883614063263], [0.7240487933158875, 1.2955412864685059]], "dtype": "float32"}]]}], ["count", {"__nd__": 8, "dtype": "int32"}]]}]}}, {"a": [{"__dict__": [["w", {"__nd__": [[-0.12729124907249822, -0.011491045993172726], [0.9978008478836363, -1.5198385772949299], [2.9836041576704964, 0.6113832765195376]], "dtype": "float64"}], ["b", {"__nd__": [-0.7730795278549334, -0.5714507211965946], "dtype": "float64"}], ["ln", {"__dict__": [["scale", {"__nd__": [1.0, 1.0], "dtype": "float64"}]]}]]}, {"__dict__": [["w", {"__nd__": [[-0.4829002926287357, -2.23150412697604], [0.40076976931951735, -1.1111909907677253], [-0.5421945982150024, -1.0959647605491671]], "dtype": "float64"}], ["b", {"__nd__": [-0.5979807974043734, 0.02380074542331151], "dtype": "float64"}], ["ln", {"__dict__": [["scale", {"__nd__": [1.0, 1.0], "dtype": "float64"}]]}]]}, {"__dict__": [["m", {"__dict__": [["w", {"__nd__": [[-1.2858316007865764, 1.516854487345044], [-1.2754771027025613, -0.49820609198173327], [-0.025620804106589196, 0.6212953773921424]], "dtype": "float64"}], ["b", {"__nd__": [0.21056887485297837, 0.03131100886795261], "dtype": "float64"}], ["ln", {"__dict__": [["scale", {"__nd__": [1.0, 1.0], "dtype": "float64"}]]}]]}], ["v", {"__dict__": [["w", {"__nd__": [[0.48547452567499033, 0.3615960430666288], [0.9805786241594653, 0.5671743404529566], [0.6659511251025941, 1.16827904685887]], "dtype": "float64"}], ["b", {"__nd__": [0.11619999732118959, 0.5278677078390953], "dtype": "float64"}], ["ln", {"__dict__": [["scale", {"__nd__": [0.21548552889828748, 0.26411950049946104], "dtype": "float64"}]]}]]}], ["count", 1000]]}, 0.0003, 0.9, 0.999, 1e-08, 0.0], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": [-0.7731702327728271, -0.5714607238769531], "dtype": "float32"}], ["ln", {"__dict__": [["scale", {"__nd__": [0.9994868636131287, 0.9995363354682922], "dtype": "float32"}]]}], ["w", {"__nd__": [[-0.1268782764673233, -0.01194135844707489], [0.9980679154396057, -1.5196614265441895], [2.9836268424987793, 0.6112840175628662]], "dtype": "float32"}]]}, {"__dict__": [["m", {"__dict__": [["b", {"__nd__": [0.12971390783786774, 0.030559983104467392], "dtype": "float32"}], ["ln", {"__dict__": [["scale", {"__nd__": [1.0, 1.0], "dtype": "float32"}]]}], ["w", {"__nd__": [[-1.2055383920669556, 1.1420186758041382], [-1.1078523397445679, -0.5595045685768127], [-0.07727818191051483, 0.44956934452056885]], "dtype": "float32"}]]}], ["v", {"__dict__": [["b", {"__nd__": [0.11644138395786285, 0.5273404717445374], "dtype": "float32"}], ["ln", {"__dict__": [["scale", {"__nd__": [0.21627004444599152, 0.26485538482666016], "dtype": "float32"}]]}], ["w", {"__nd__": [[0.4852222502231598, 0.3662140667438507], [0.9797586798667908, 0.5678419470787048], [0.6655791401863098, 1.1683119535446167]], "dtype": "float32"}]]}], ["count", {"__nd__": 1001, "dtype": "int32"}]]}]}}, {"a": [{"__dict__": [["emb", {"__nd__": [[-0.6345945527579017, 0.2830374182794798, -0.09478610504143659], [-1.6627133692477325, 1.5091370762392131, -0.043169049428652025], [-0.19199754889697485, 0.8128328128899321, -1.4437899886371925], [-0.7307507940363667, 0.8894061963160548, -1.9597103663260775]], "dtype": "float64"}], ["bias", {"__nd__": 0.5, "dtype": "float64"}]]}, {"__dict__": [["emb", {"__nd__": [[-0.9028946741008395, 1.6072287561169414, 0.5499243895932716], [-1.0300367909323753, 0.00491004335618158, 0.6954755771197471], [-0.8377708956038068, -0.026600237910753565, 0.3046159862046389], [-0.001216765808515476, 1.4873194940159187, 0.07347588150413752]], "dtype": "float64"}], ["bias", {"__nd__": -1.0, "dtype": "float64"}]]}, {"__dict__": [["m", {"__dict__": [["emb", {"__nd__": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float64"}], ["bias", {"__nd__": 0.0, "dtype": "float64"}]]}], ["v", {"__dict__": [["emb", {"__nd__": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float64"}], ["bias", {"__nd__": 0.0, "dtype": "float64"}]]}], ["count", 0]]}, 0.01, 0.8, 0.9, 1e-08, 0.5], "e": {"__tup__": [{"__dict__": [["bias", {"__nd__": 0.5099999904632568, "dtype": "float32"}], ["emb", {"__nd__": [[-0.6214215755462646, 0.2716222405433655, -0.10431218147277832], [-1.644399881362915, 1.4915913343429565, -0.05295320600271225], [-0.1810375601053238, 0.8187686800956726, -1.4465709924697876], [-0.7170971035957336, 0.8749591708183289, -1.959911823272705]], "dtype": "float32"}]]}, {"__dict__": [["m", {"__dict__": [["bias", {"__nd__": -0.20000000298023224, "dtype": "float32"}], ["emb", {"__nd__": [[-0.18057893216609955, 0.3214457631111145, 0.1099848747253418], [-0.20600736141204834, 0.0009820087580010295, 0.13909511268138885], [-0.16755418479442596, -0.005320047494024038, 0.060923196375370026], [-0.00024335317721124738, 0.29746389389038086, 0.014695176854729652]], "dtype": "float32"}]]}], ["v", {"__dict__": [["bias", {"__nd__": 0.10000000149011612, "dtype": "float32"}], ["emb", {"__nd__": [[0.08152187615633011, 0.2583184242248535, 0.03024168126285076], [0.10609757900238037, 2.4108528577926336e-06, 0.04836862534284592], [0.07018601149320602, 7.075726171024144e-05, 0.00927908904850483], [1.4805191028699483e-07, 0.22121192514896393, 0.0005398705252446234]], "dtype": "float32"}]]}], ["count", {"__nd__": 1, "dtype": "int32"}]]}]}}, {"a": [{"__dict__": [["w", {"__nd__": [[-0.1933009137039584, 0.18703909749522762], [-0.4171161481392536, -0.8450447926057556], [-0.2704161112651102, -0.6779793876791749]], "dtype": "float64"}], ["b", {"__nd__": [1.8146614462614652, 2.1164273239449556], "dtype": "float64"}], ["ln", {"__dict__": [["scale", {"__nd__": [1.0, 1.0], "dtype": "float64"}]]}]]}, {"__dict__": [["w", {"__nd__": [[0.0, 0.0], [0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}], ["b", {"__nd__": [0.0, 0.0], "dtype": "float64"}], ["ln", {"__dict__": [["scale", {"__nd__": [0.0, 0.0], "dtype": "float64"}]]}]]}, {"__dict__": [["m", {"__dict__": [["w", {"__nd__": [[-0.4034036141603932, -2.8174390055978416], [-0.3863037573387881, 0.6481309753606957], [-0.6464698101790399, 0.6671902377038638]], "dtype": "float64"}], ["b", {"__nd__": [0.6191705105552058, -0.635104880790905], "dtype": "float64"}], ["ln", {"__dict__": [["scale", {"__nd__": [1.0, 1.0], "dtype": "float64"}]]}]]}], ["v", {"__dict__": [["w", {"__nd__": [[0.2566040818113128, 0.9951539478620052], [0.4357721270735068, 1.1175727198767558], [0.5409938104831117, 1.4676663621979171]], "dtype": "float64"}], ["b", {"__nd__": [0.3204907601460438, 1.0866886104578148], "dtype": "float64"}], ["ln", {"__dict__": [["scale", {"__nd__": [0.3195667036908708, 1.5016512245910414], "dtype": "float64"}]]}]]}], ["count", 3]]}, 0.1, 0.9, 0.999, 1e-08, 0.2], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": [1.7965635061264038, 2.1265087127685547], "dtype": "float32"}], ["ln", {"__dict__": [["scale", {"__nd__": [0.9707283973693848, 0.98649662733078], "dtype": "float32"}]]}], ["w", {"__nd__": [[-0.17625731229782104, 0.23003274202346802], [-0.399090439081192, -0.8382889032363892], [-0.25046393275260925, -0.6735328435897827]], "dtype": "float32"}]]}, {"__dict__": [["m", {"__dict__": [["b", {"__nd__": [0.5572534203529358, -0.5715944170951843], "dtype": "float32"}], ["ln", {"__dict__": [["scale", {"__nd__": [0.8999999761581421, 0.8999999761581421], "dtype": "float32"}]]}], ["w", {"__nd__": [[-0.363063246011734, -2.5356950759887695], [-0.34767335653305054, 0.5833178162574768], [-0.5818228125572205, 0.6004711985588074]], "dtype": "float32"}]]}], ["v", {"__dict__": [["b", {"__nd__": [0.32017025351524353, 1.0856019258499146], "dtype": "float32"}], ["ln", {"__dict__": [["scale", {"__nd__": [0.31924712657928467, 1.5001494884490967], "dtype": "float32"}]]}], ["w", {"__nd__": [[0.2563474774360657, 0.9941588044166565], [0.4353363513946533, 1.116455078125], [0.5404528379440308, 1.4661986827850342]], "dtype": "float32"}]]}], ["count", {"__nd__": 4, "dtype": "int32"}]]}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(adamw_update, json.loads(_TESTS), 'jax', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")